## Development Environment for Model Building

This notebook serves as our development environment and is divided into different phases:

**Phase 1 — Data Understanding & Cleaning**
1. Load dataset, confirm shape/columns
2. Check missing values, decide handling per column
3. Explore class balance (`queue` counts, `priority` counts, `language` split)
4. Combine `subject` + `body` into a single `text` field
5. Basic text stats (length distribution, duplicates check)

**Phase 2 — Sentiment Label Generation (since it doesn't exist in the data)**

6. Pick a local pretrained sentiment model (e.g. multilingual BERT-based)
7. Run it over the full dataset to generate sentiment labels
8. Pull a random sample (~250-300 rows), manually label as a team
9. Compare model labels vs human labels, compute agreement (Cohen's kappa)
10. Sanity-check generated sentiment against `priority` (Critical should skew more negative)
11. Decide: keep as-is, or refine the rubric/model if agreement is weak

**Phase 3 — Train/Test Prep**

12. Stratified train/validation/test split (by `queue`, ideally checking `sentiment` balance too)
13. Decide feature representation: TF-IDF vs multilingual sentence embeddings
14. Turn `text` into features on train/val/test consistently (fit on train only)

**Phase 4 — Modeling**

15. Train baseline **intent model** (text → `queue`), evaluate (F1 per class, confusion matrix)
16. Train baseline **sentiment model** (text → sentiment), evaluate similarly
17. Check performance by language (English vs German) — catch any imbalance in accuracy
18. (Stretch) Train a small fine-tuned transformer, compare vs classical baseline
19. Pick final models, save them (`joblib`/model folder)

**Phase 5 — Turn Notebook Into Code**

20. Refactor cleaning + feature prep into reusable functions/scripts
21. Wrap into a single `classify_and_route(text, language)` function
22. Save trained models to disk in a form the app can load


In [14]:
#importing libraries
import pandas as pd
import numpy as np
from datasets import load_dataset


In [15]:
#loading datasets
ds = load_dataset("Tobi-Bueck/customer-support-tickets")
df = ds["train"].to_pandas()

df.shape

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets/resolve/ddf1c81a5475992c4fa6752bf1e8b4e31f07bbeb/customer-support-tickets.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since Tobi-Bueck/customer-support-tickets couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\hannah.igboke\.cache\huggingface\datasets\Tobi-Bueck___customer-support-tickets\default\0.0.0\ddf1c81a5475992c4fa6752bf1e8b4e31f07bbeb (last modified on Thu Aug 27 19:39:27 2026).


(61765, 16)

In [16]:
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [17]:
#selecting columns of interest
df_2 = df[['subject', 'body', 'queue', 'priority', 'language']]
df_2.head()

,subject,body,queue,priority,language
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Technical Support,high,de
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,high,en
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,medium,en
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,low,en
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,medium,en


### Data Cleaning

In [18]:
#checking for missing values
df_2.isnull().sum()

subject     5299
body           2
queue          0
priority       0
language       0
dtype: int64

Analysing the missing values and possible next steps:

- The data has about 5,299 rows that are missing a subject. What approach would be best here to handle missing values? Since we'll be concatenating the message body to the subject, for cases where the subject is misisng we can replace it with a an empty string so that the concatenation still works.
- There are two instances where the body of the message is missing, we would need to drop those rows given that the body is a major part of what the model is expected to learn from.


In [19]:
#checking data types
df_2.dtypes

subject     object
body        object
queue       object
priority    object
language    object
dtype: object

## Exploratory Data Analysis

In [20]:
df_2['type'].value_counts()

KeyError: 'type'

In [ ]:
#the distribution of the different teams we'll be routing the messages to
df_2['queue'].value_counts()

queue
Technical Support                         14186
Product Support                            8960
Customer Service                           7420
IT Support                                 5725
Billing and Payments                       4874
Returns and Exchanges                      2438
Service Outages and Maintenance            1912
Sales and Pre-Sales                        1490
Human Resources                             914
General Inquiry                             668
Pets & Animals/Pet Services                 386
News                                        383
IT & Technology/Security Operations         365
Autos & Vehicles/Sales                      364
Health/Medical Services                     362
Home & Garden/Home Improvement              361
Pets & Animals/Veterinary Care              356
Health/Mental Health                        347
Business & Industrial/Manufacturing         346
Online Communities/Forums                   343
Shopping/E-commerce               

In [ ]:
#distribution of the message priority
df_2['priority'].value_counts()

priority
medium      23378
high        21925
low         12765
critical     1914
very_low     1783
Name: count, dtype: int64

In [ ]:
#distribution of langauges in the datasets
df_2['language'].value_counts()

language
de    33504
en    28261
Name: count, dtype: int64

We have a bit more german messages compared to english messages. There are two posisble approaches here:
- We translate every german ticket to english before it is used in training
- We retain the german language tickets as is and use multilingual embeddings

The most viable option is going with option 2. In option 1, the downside is that:

- Adds a translation model/step as a dependency
- Translation quality varies and can distort tone/sentiment (translation often flattens emotional nuance which is exactly the signal our sentiment model needs).